# ZMQ: OCUDU CU-CP, CU-UP and DU with an srsUE

An Open5GS core, an OCUDU RAN split into CU-CP, CU-UP and DU, and an srsUE, linked over ZeroMQ instead of radio. The extra section switches the split from E1 to F1 on the running network ([OCUDU ZeroMQ tutorial](https://docs.ocudu.org/tutorials/srsue/#zeromq-based-setup)).

- **Monitoring** (NADOR) collects RAN KPIs, N3 throughput per UE and slice, TCP round-trip time per UE and slice at the gNB and UPF, and the logs of the RAN, core and monitoring pods; the profile is [`monitoring.yaml`](monitoring.yaml).
- **Infrastructure:** VMs on SLICES; no radio hardware.

Run the cells in order; deployment uses real resources. Monitoring is deployed after the RAN and before the UEs attach, so its sniffers see every session being set up.

## 1. Install

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal servers:

In [ ]:
%pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
%pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Install the IFRAN:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../../..").resolve()
# Nador installs the supported IFRAN release.
%pip install --upgrade nador
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


## 2. Describe the network

In [ ]:
from metamodel_3gpp.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  sim_ru_du:
    txGainDB: 0
    rxGainDB: 0

  sim_ru_ue:
    txGainDB: 50
    rxGainDB: 40

ran:
  components:
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
      config:
        cu_cp: cucp
    du:
      type: du
      config:
        ru: sim_ru_du
        cu_cp: cucp
        cu_up: cuup 
        radioConfiguration:
          nrBand: 3
          downlinkArfcn: 368500
          channelBandwidthMhz: 10
          subcarrierSpacingKhz: 15
          ssbPeriodicityMs: 10
          prachConfigurationIndex: 1

ues:
  sim_ue:
    config:
      nrBands: [3]
      ru: sim_ru_ue
      gnbs: [du]

slices:
  internet:
    sst: 1
    sd: 0xFFFFFF # explicitly set sd to avoid core regection
"""
)

In [ ]:
from ifran.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    project: post5g-beta
    experiment: 5gbp_exp
    flavor: xlarge # ocudu zmq needs more than 5 CPU cores (see: https://github.com/srsran/srsRAN_Project/issues/1472)
    extra_path: "" # Use SLICES CLI from PATH

ran:
  implementation: ocudu
  components:
    cucp:
      type: cu_cp
      advanced_config:
        inactivity_timer: 7200
    du:
      type: du
      advanced_config:
        srate: 11.52
        base_srate: "11.52e6"
        pdcch:
          common:
            ss0_index: 0
            coreset0_index: 6
          dedicated:
            ss2_type: common
            dci_format_0_1_and_1_1: false
        pdsch:
          mcs_table: qam64
        pusch:
          mcs_table: qam64

ues:
  sim_ue:
    type: containerized
    implementation: srsue
    advanced_config:
      srate: "11.52e6"
"""
)

## 3. Provision the machines

In [ ]:
from ifran.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision()

## 4. Kubernetes

In [ ]:
from ifran.deployment import K8SManagement

k8s = K8SManagement(prov)

In [ ]:
await k8s.deploy()

## 5. Core

In [ ]:
from ifran.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## 6. RAN

In [ ]:
from ifran.deployment import RANManagement

ran = RANManagement(core)

from nador import MonitoringDeployment, configure_ran_monitoring
ran_kpi_config = MonitoringDeployment.from_yaml_file("monitoring.yaml")
configure_ran_monitoring(ran, ran_kpi_config)

In [ ]:
configure_ran_monitoring(ran, ran_kpi_config)
await ran.deploy()

Get OCUDU ZMQ DU logs:

In [ ]:
print(await ran.get_logs(quiet=True))

## 7. Monitoring (before the UEs attach)

The profile uses the control host selected by Kubernetes, including physical
Sopnodes. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from nador import MonitoringManagement
from nador import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

## 8. UEs

In [ ]:
from ifran.deployment.ue import UEManagement

ue = UEManagement(preceding=ran)

In [ ]:
await ue.deploy()

Get SRSUE ZMQ UE logs:

In [ ]:
print(await ue.get_logs(quiet=True))

Get DU logs after UE was connected:

In [ ]:
print(await ran.get_logs(quiet=True))

Verify that the UE got a PDU session:

In [ ]:
print((await ue.run_command(command=["ip", "addr"], quiet=True, flatten=False))["sim_ue"])


### Test connectivity

In [ ]:
print((await ue.run_command(command=["apt", "update"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print((await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print(
    (await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


In [ ]:
print(
    (await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


## 9. Extra: switch from an E1 to an F1 split

We can change the OCUDU RAN's split from E1 to F1 while keeping the already deployed Open5GS core.

To do so, remove the deployed UE and RAN:

In [ ]:
await monitoring.remove()
await ue.remove(quiet=True)
await ran.remove(quiet=True)

Remove existing OCUDU E1s-split RAN components:

In [ ]:
ran.remove_component("cuup")
ran.remove_component("cucp")
ran.remove_component("du")

Add CU that will replace the previous CU-CP and CU-UP:

In [ ]:
from metamodel_3gpp.models.ran import CUComponent, RANComponentType
from ifran.models.ran import DeployedRANComponent

ran.add_component(
    name="cu",
    component_meta=CUComponent(type=RANComponentType.CU),
    component_deployment=DeployedRANComponent.from_yaml(
"""
type: cu
implementation: ocudu
advanced_config:
  inactivity_timer: 7200
"""
    ),
)

Update and add the F1-split RAN DU from the previous one:

In [ ]:
du_meta = metamodel.ran.components["du"]
du_deployemnt = deployment_model.ran.components["du"]

# update cu_cp and cu_up references in du component
du_meta.config.cu_cp = "cu"
du_meta.config.cu_up = "cu"

ran.add_component(
    name="du",
    component_meta=du_meta,
    component_deployment=du_deployemnt,
)

Deploy the OCUDU F1-split RAN:

In [ ]:
configure_ran_monitoring(ran, ran_kpi_config)
await ran.deploy()

In [ ]:
# The second layout replaces CU-CP/CU-UP with a combined CU.
monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)
packet_target = monitoring.deployment_model.monitoring.targets[-1]
packet_target.component = "cu"
packet_target.container = "cu"
await monitoring.deploy()


Redeploy the SRSUE based on the changed RAN:

In [ ]:
ue = UEManagement(preceding=ran)

In [ ]:
await ue.deploy()

Get DU logs after UE was connected:

In [ ]:
print(await ran.get_logs(quiet=True))

Verify that the UE got a PDU session:

In [ ]:
print((await ue.run_command(command=["ip", "addr"], quiet=True, flatten=False))["sim_ue"])


### Test connectivity

In [ ]:
print((await ue.run_command(command=["apt", "update"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print((await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print(
    (await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


In [ ]:
print(
    (await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


## 10. Traffic and experiment

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline.

In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun, iperf_summary
from ifran.deployment import ProvisioningManagement
from nador.traffic import Iperf

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = ["sim_ue"]
UE_INTERFACE = "tun_srsue"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

clients = ue
iperf = await Iperf.setup(clients, server=SERVER_NODE, interface=UE_INTERFACE, clients=CLIENTS)

print(f"Server: {SERVER_NODE} ({iperf.server_ip}); UE clients: {iperf.ue_ips}")


`iperf.serve(port)` starts a one-shot iperf3 server on the server node. `iperf.client(ue, direction, seconds, port)` runs the UE side (`downlink` adds iperf's reverse mode) and returns iperf3's JSON report, with an `error` key when the test failed. `iperf.run(ue, direction, seconds)` does both on a fresh port.

In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'sim_ue', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'sim_ue', "direction": "downlink", "duration": DURATION},
]

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        await iperf.serve(port)

        with experiment.section(test["name"]):
            report = await iperf.client(
                client=test["client"],
                direction=test["direction"],
                duration=test["duration"],
                port=port,
            )

        experiment.save_json(f"{test['name']}.json", report)
        if report.get("error"):
            raise RuntimeError(f"{test['name']}: {report['error']}")

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")
print(iperf_summary(output_directory))